# AnyTime Universal Benchmark

Run early-exit benchmark for any AnyTime model. Switch by changing **one import line** below.

All HW metrics **per-PID** (process-scoped). Energy = `Î£ (power_w_i Ã— end_to_end_sec_i)` in Joules.

Per-run dir contains:
- `hw_results.json` â€” latency + memory + energy + per-PID GPU/CPU + FLOPs/params
- `quality_results.json` â€” accuracy / F1 / mAP / PPL (only if `skip_quality=False`)

Default sweep = HW-only. Pass `skip_quality=False` to enable quality eval.

Outputs:
- `logs/benchmark/{model}/...` â€” per-run JSONs (nested 2-3 levels deep)
- `results/{model}/` â€” CSV summaries


## 1. Setup

In [ ]:
import sys
import os
from pathlib import Path

REPO_ROOT = Path(".").resolve()
sys.path.insert(0, str(REPO_ROOT))
print("repo root:", REPO_ROOT)

In [ ]:
# HF login (only needed if pulling from a private repo)
from shared import load_env

load_env()

try:
    from google.colab import userdata

    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN", ""))
except Exception:
    pass

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login

    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("HF: logged in")
else:
    print("HF: no token, public repos only")

## 2. Choose model

**Change ONE import line below** to swap which model gets benchmarked.

In [ ]:
# === SWAP THIS LINE TO PICK MODEL ===
from benchmark_config import bert as cfg
# from benchmark_config import llama  as cfg
# from benchmark_config import vision as cfg
# from benchmark_config import yolo   as cfg

print("model :", cfg.NAME)
print("family:", cfg.MODEL_FAMILY)
print("out   :", cfg.OUT_DIR)

## 3. Run full sweep

Default = HW-only (`skip_quality=True`). Each run does:
- `profile_hw()` â€” HW + latency + per-PID power/VRAM/CPU + FLOPs/params + EDP
- `evaluate_quality()` â€” (skipped by default)

Override sweep dims via `only_*` kwargs (see `cfg.run_all` signature).


In [ ]:
cfg.run_all()                            # default: HW-only sweep
# cfg.run_all(skip_quality=False)        # also run quality eval
# cfg.run_all(only_weight_source='pretrained', only_exit=4)  # single-point smoke


## 4. Export CSVs

Recursive walk over per-run dirs (handles 2-3 level nesting per model). Merges
`hw_results.json` + `quality_results.json` if both exist. Groups by parent dir.


In [ ]:
from shared import write_benchmark_csvs
from collections import defaultdict

out_dir = Path(cfg.OUT_DIR)
csv_root = REPO_ROOT / 'results' / cfg.NAME
csv_root.mkdir(parents=True, exist_ok=True)

# Find every run dir (contains hw_results.json), group by parent
run_dirs = [p.parent for p in out_dir.rglob('hw_results.json')]
print(f'found {len(run_dirs)} run dirs under {out_dir}')

groups = defaultdict(dict)
for rd in run_dirs:
    rel_parent = rd.parent.relative_to(out_dir).as_posix()
    group_key = rel_parent.replace('/', '_') or 'root'
    groups[group_key][rd.name] = rd

for group_key, runs in sorted(groups.items()):
    csv_dir = csv_root / group_key
    csv_dir.mkdir(parents=True, exist_ok=True)
    write_benchmark_csvs(
        results_files=runs,
        out_dir=csv_dir,
        baseline_key=None,
        method_order=sorted(runs.keys()),
    )
    print(f'  wrote {len(runs)} runs -> {csv_dir}')

print('CSVs under:', csv_root)


## 5. Quick view

In [ ]:
import pandas as pd

for csv_name in ['latency.csv', 'energy.csv', 'quality.csv', 'hardware.csv']:
    for f in csv_root.rglob(csv_name):
        rel = f.relative_to(csv_root)
        print(f'=== {rel} ===')
        print(pd.read_csv(f).head(8))
        print()
